# 무역 법리 판단 파인튜닝 데이터 생성 엔진 (v3.3)

이 노트북은 `trade_law_pipeline_prompts.md` 가이드라인을 100% 준수하여 고품질의 법률 AI 학습 데이터를 생성합니다.

### 0. 환경 설정
API 키 로드 및 데이터를 저장할 기본 디렉토리를 설정합니다.

In [1]:
from datetime import datetime
import os
import json
import asyncio
from pathlib import Path
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
MODEL_ID = 'gemini-3.1-flash-lite' # 'gemma-4-31b-it', 'gemini-3.1-flash-lite'

BASE_DIR = Path("data/finetune_gen")
SOURCE_DIR = BASE_DIR / "source"
SCRIPT_DIR = BASE_DIR / "script"
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
SCRIPT_DIR.mkdir(parents=True, exist_ok=True)

print(f"[System] 준비 완료. 모델: {MODEL_ID}")

[System] 준비 완료. 모델: gemini-3.1-flash-lite


### 1. 기본 법령 및 배치 설정
학습 데이터의 근거가 될 법령 리스트와 생성할 샘플의 구성을 정의합니다.

In [2]:
INSTRUCTION_A = """아래 무역 계약서 초안을 검토하여 잠재적 법적 리스크를 분석하라.
판단 시 다음 순서를 반드시 따르라:
(1) 적용 법령 및 조문 특정 — Article/조 번호 명기 필수
(2) 법령 간 충돌·누락 확인, 우선순위 적용, 복수 법령 결합 시 관계 유형 명시
(3) 조항별 리스크 분석 — 리스크 유형(무효 가능성 / 집행 불능 / 분쟁 유발 / 불리한 조건) 명시
(4) 직접 조문이 없는 경우 유사 조문 유추→상위 원칙→실무 관행 순으로 폴백 추론하고 추론 등급 명시
(5) 수정 권고안 — Before(현행 조항 요지) → After(권고 조항 요지) 형식으로 제시
(6) 최종 판정: 🔴위험(즉시 수정 필요) / 🟡주의(수정 권고) / 🟢적합(현행 유지 가능)
(7) 판단 불가 시: 1~4단계 폴백 추론 모두 시도 후에도 불가한 경우에만 선언, 필요 정보 명시"""
INSTRUCTION_B = """아래 무역서류 간 불일치 및 관련 법령 위반 여부를 판단하라.
판단 시 다음 순서를 반드시 따르라:
(1) 적용 법령 및 조문 특정 — Article/조 번호 명기 필수
(2) 법령 간 충돌 확인, 우선순위 적용, 복수 법령 결합 시 관계 유형 명시
(3) 서류별 불일치 특정 — 서류명, 항목, 수치를 구체적으로 명시
(4) 법적 하자 여부 판단 — 직접 조문 없을 시 유사 조문 유추→상위 원칙→실무 관행 순으로 폴백 추론하고 추론 등급 명시
(5) 매도인·매수인·은행 등 관계자 항변 검토 및 기각/인용 근거
(6) 조치 가능성 열거 — 서류 보완(amendment), 거절(rejection), 권리 유보 등
(7) 최종 판정: 🔴하자(거절·해제 가능) / 🟡조건부(보완 시 수리 가능) / 🟢적합(하자 없음)
(8) 판단 불가 시: 1~4단계 폴백 추론 모두 시도 후에도 불가한 경우에만 선언, 필요 정보 명시"""

### 1-1. 법령 리스트 및 배치 파라미터 설정
`LAW_LIST`는 세 프롬프트 모두에 주입되는 핵심 상수입니다. 영국법 시리즈 추가 반영 (v3.3).

In [3]:
LAW_LIST = """
- CISG (UN 국제물품매매계약협약)
- UCP 600 (신용장통일규칙)
- eUCP (신용장통일규칙 전자적 제시 보충규칙)
- ISBP 821 (국제표준은행관행)
- Incoterms 2020
- 대외무역법 (대한민국)
- 외국환거래법 (대한민국)
- 독점규제 및 공정거래에 관한 법률 (대한민국)
- 약관의 규제에 관한 법률 (대한민국)
- 상법 (대한민국)
- 중재법 (대한민국)
- Arbitration Act 1996 (British)
- Carriage of Goods by Sea Act 1992, COGSA 1992 (British)
- Contracts (Rights of Third Parties) Act 1999 (British)
- Electronic Trade Documents Act 2023, ETDA 2023 (British)
- Sale of Goods Act 1979, SGA 1979 (British)
- Unfair Contract Terms Act 1977, UCTA 1977 (British)
- Institute Cargo Clauses, ICC (A) / (B) / (C)
- Insurance Act 2015 (British)
- Marine Insurance Act 1906, MIA 1906 (British)
""".strip()

CONFIG = {
    "FORMAT_TYPE":        "B",      # "A" / "B" / "A+B"
    "SAMPLE_COUNT":       20,         # 총 생성 샘플 수
    "CASE_TYPES":         "자동 분배", # "A1~A8 / B1~B8" 또는 "자동 분배"
    "JURISDICTION":       "한국 당사자(매도인 또는 매수인) ↔ 랜덤 국가", # 예: "한국 매도인 ↔ 영국 매수인" 또는 "자동 분배"
    'INPUT_LANG':        'mixed',   # 'ko' | 'en' | 'mixed'
    "EXTRA_CONSTRAINTS": """
        계약서 본문 및 서류 대조표는 영어로 작성하라.
        relation_required=true 케이스는 반드시 3개 이상의 법령이 교차하는 시나리오로 설계하라.
        fallback_level=3 이상 케이스는 상충하는 판례/관행을 2개 이상 제시하라.
        모든 케이스에 분쟁 발생 타임라인(날짜 순서)을 포함하라.
    """,
}

# INPUT_LANG별 P2 언어 지침 (P2_SYS {{LANG_CONSTRAINT}}에 자동 주입)
LANG_CONSTRAINTS = {
    'ko': '',
    'en': (
        '모든 텍스트를 영어로 작성하라. '
        '구조 레이블(Contract Overview, Actual Situation 등)도 영어로 표기한다. '        
    ),
    'mixed': (
        '구조 레이블과 항목명은 한국어로, '
        '계약 조항 원문·수치·날짜·서류명은 영어로 작성하라. '        
    ),
}

print("[LAW_LIST 로드 완료]")
print(f"[CONFIG] FORMAT={CONFIG['FORMAT_TYPE']} | SAMPLES={CONFIG['SAMPLE_COUNT']} | JX={CONFIG['JURISDICTION']}")


[LAW_LIST 로드 완료]
[CONFIG] FORMAT=B | SAMPLES=20 | JX=한국 당사자(매도인 또는 매수인) ↔ 랜덤 국가


### 2. [Step 1] 시나리오 기획 프롬프트 정의

In [4]:
P1_SYS = r"""당신은 국제 무역 법률 전문가이자 파인튜닝 데이터 엔지니어입니다.
아래 지시에 따라 파인튜닝 샘플의 시나리오 메타데이터를 설계하십시오.

[적용 법령]
{{LAW_LIST}}

[설계 규칙]
1. 판정 등급 배분 (포맷별 독립 적용)
   - 🔴 위험/하자: 30%
   - 🟡 주의/조건부: 40%
   - 🟢 적합: 30%
   - 🟢 적합이 0개인 배치는 허용하지 않는다.

2. 추가 배치 비율 (판정 등급과 독립)
   - relation_required=true 케이스: 전체의 50% 이상
   - fallback_level≥2 케이스: 전체의 30% 이상
   - 두 조건은 동일 케이스에 중복 산입 가능.

3. 포맷 A 케이스 유형: A1~A8 중 선택
   A1: 인도 조건 및 위험 이전 조항 리스크
   A2: 결제 조건 및 L/C 요건 누락 리스크
   A3: 준거법·분쟁해결 조항 집행 불능 리스크
   A4: 보험 조항 부보 의무 불명확 리스크
   A5: 국내 강행법규 저촉 리스크
   A6: 면책·불가항력 조항 과대·과소 설정 리스크
   A7: 품질 기준 조항 모호성 리스크
   A8: 계약 해제·손해배상 조항 불균형 리스크

4. 포맷 B 케이스 유형: B1~B8 중 선택
   B1: L/C 조건과 선적서류(B/L, Invoice) 불일치
   B2: 보험증권 조건 불일치
   B3: 선적일·유효기일 초과
   B4: 수량·금액 수치 불일치
   B5: 원산지증명서 또는 검사증명서 누락·불일치
   B6: 수출입 규제·허가서류 미비
   B7: 서류 통지 기간 준수 여부
   B8: 복수 서류 간 당사자명·주소 불일치

5. 필수 포함 관계 유형 (relation_required=true 케이스에 사용)
   - 적용 제한: 법령 A가 특정 조건에서만 유효
   - 상위 원칙 포섭: 하위 규칙 흠결을 상위 법령이 메움
   - 교차 충돌: 두 법령이 동일 사항에 상반된 의무 부과
   - 요건 보완: 법령 A 요건을 법령 B가 구체화
   - 절차 연동: 법령 A 판단이 법령 B 절차를 촉발


6. fallback_level 정의
   1: 직접 조문 적용 (완전 일치 조문 존재)
   2: 유사 조문 유추 적용
   3: 상위 법 원칙 적용
   4: 실무 관행 기반 의견 제시

7. UK 관할 케이스 설계 지침
    - UK 관할 케이스는 영국법 시리즈(Arbitration Act 1996, COGSA 1992,
     SGA 1979, UCTA 1977, MIA 1906, Insurance Act 2015, ETDA 2023)를
     활용하여 설계할 수 있다.
   - 한국 관할 케이스와 UK 관할 케이스가 동일 배치에 혼재할 때,
     relation_required=true 케이스 중 최소 1건은 두 관할 법령이 교차하는 시나리오를 설계하라.

[출력 형식]
반드시 순수 JSON 배열만 출력하십시오. 마크다운 코드블록, 설명 문장, 전처리 텍스트 일체 금지.

스키마:
[
  {
    "id": "string",           // 예: "A-001", "B-003"
    "format": "A" | "B",
    "case_type": "string",    // 예: "A1", "B4"
    "verdict": "🔴" | "🟡" | "🟢",
    "relation_required": boolean,
    "relation_types": ["string"],   // relation_required=false이면 []
    "fallback_level": 1 | 2 | 3 | 4,
    "jurisdiction": "string",       // 예: "한국 매도인 ↔ 영국 매수인"
    "commodity": "string",          // 품목 + HS Code 예시
    "key_conflict": "string"        // 핵심 쟁점 한 줄 요약
  }
]"""
P1_USR = r"""아래 파라미터로 시나리오 메타데이터를 생성하십시오.

[GENERATE]
- 포맷 타입: {{FORMAT_TYPE}}
- 생성 샘플 수: {{SAMPLE_COUNT}}개
- 케이스 유형: {{CASE_TYPES}}
- 관할 국가: {{JURISDICTION}}
- 추가 제약: {{EXTRA_CONSTRAINTS}}

배치 비율 검증을 수행하고 결과 JSON만 출력하십시오."""


### 3. [Step 1] 시나리오 기획 실행

In [51]:
def setup_scenarios():
    sys = P1_SYS.replace("{{LAW_LIST}}", LAW_LIST)
    usr = P1_USR.replace("{{FORMAT_TYPE}}", CONFIG['FORMAT_TYPE'])\
                .replace("{{SAMPLE_COUNT}}", str(CONFIG['SAMPLE_COUNT']))\
                .replace("{{CASE_TYPES}}", CONFIG['CASE_TYPES'])\
                .replace("{{JURISDICTION}}", CONFIG['JURISDICTION'])\
                .replace("{{EXTRA_CONSTRAINTS}}", CONFIG['EXTRA_CONSTRAINTS'])
    
    res = client.models.generate_content(model=MODEL_ID, contents=[sys + "\n" + usr], 
                                         config={'response_mime_type': 'application/json', "max_output_tokens": 65536})
    metas = json.loads(res.text)
    # 현재 실행 시간을 기반으로 고유 접미사 생성
    run_id = datetime.now().strftime('%m%d_%H%M')
    for m in metas:
        # ID 뒤에 run_id를 붙여 덮어쓰기 방지
        s_dir = SOURCE_DIR / f"{m['id'].lower().replace(' ', '')}_{run_id}"
        s_dir.mkdir(parents=True, exist_ok=True)
        with open(s_dir / "meta.json", "w", encoding="utf-8") as f: json.dump(m, f, indent=4, ensure_ascii=False)
    print(f"✅ {len(metas)}개 시나리오 준비 완료. (Run ID: {run_id})")
    return metas, run_id

metadata_list, run_id = setup_scenarios()

✅ 20개 시나리오 준비 완료. (Run ID: 0515_0111)


In [ ]:
# Step 1에서 출력된 Run ID를 자동으로 가져오거나, 
# 과거에 생성한 특정 Run ID를 대상으로 하고 싶을 때 수정하세요.

# 1. 자동으로 최근 생성된 run_id 사용 (Step 1 함수 수정이 필요할 수 있음)
# 만약 setup_scenarios에서 run_id를 리턴하지 않는다면 아래와 같이 수동 지정도 가능합니다.
# TARGET_RUN_ID = "0507_1130" 

# 2. None으로 설정하면 전체 폴더를 대상으로 기존처럼 동작합니다.
TARGET_RUN_ID = run_id if 'run_id' in locals() else None
TARGET_RUN_ID = None
print(f"[System] 현재 작업 대상 Run ID: {TARGET_RUN_ID if TARGET_RUN_ID else '전체 대상'}")

[System] 현재 작업 대상 Run ID: 0515_0111


### 4. [Step 2] 사실관계(Input) 생성 프롬프트 정의

In [7]:
P2_SYS_KO_LABELS = """
[포맷 A input 작성 규칙 — format="A"인 경우]
아래 6개 항목 블록을 모두 포함하십시오. 불명확한 항목은 "[미상]"으로 표기.
 
  【계약 개요】
  - 준거법 후보:
  - 계약 당사자: (매도인 국가 ↔ 매수인 국가, 각 법령 체약국 여부)
  - 계약 체결 예정일:
  - 품목: (HS Code 포함)
  - 계약 금액 및 통화:
  - 인도 조건 후보: (Incoterms 2020 기준)
 
  【계약서 초안 주요 조항】
  - 품질 기준 조항:
  - 인도 및 선적 조항:
  - 결제 조건 조항:
  - 보험 조항:
  - 준거법·분쟁해결 조항:
  - 기타 특약 조항:
 
[포맷 B input 작성 규칙 — format="B"인 경우]
아래 두 블록을 모두 포함하십시오. 서류 대조표 항목은 최소 3개.
 
  【계약 개요 (기준선)】
  - 준거법:
  - 계약 당사자:
  - 계약 체결일:
  - 품목: (HS Code 포함)
  - 계약 금액 및 통화:
  - 인도 조건: (Incoterms 2020 기준)
  - 선적/인도 기일:
  - 품질·수량 기준:
 
  【실제 발생 사항】
  - 서류 대조표:
    | 항목 | 계약서·L/C 기준 | 실제 서류 내용 | 서류명 |
    |------|----------------|---------------|--------|
    | ... | ... | ... | ... |
  - 통지 일자 및 방법:
  - 청구 내용:
  - 관계자 항변: (매도인 / 매수인 / 은행 각각)
"""
 
P2_SYS_EN_LABELS = """
[Format A input rules — when format="A"]
Include all 6 blocks below. Mark unclear items as "[Unknown]".
 
  [Contract Overview]
  - Governing Law Candidates:
  - Contracting Parties: (Seller country ↔ Buyer country, signatory status to applicable conventions)
  - Expected Contract Date:
  - Commodity: (including HS Code)
  - Contract Amount & Currency:
  - Delivery Term Candidates: (per Incoterms 2020)
 
  [Key Draft Contract Clauses]
  - Quality Standard Clause:
  - Delivery & Shipment Clause:
  - Payment Terms Clause:
  - Insurance Clause:
  - Governing Law & Dispute Resolution Clause:
  - Special Provisions:
 
[Format B input rules — when format="B"]
Include both blocks below. Discrepancy table must have at least 3 items.
 
  [Contract Overview (Baseline)]
  - Governing Law:
  - Contracting Parties:
  - Contract Date:
  - Commodity: (including HS Code)
  - Contract Amount & Currency:
  - Delivery Term: (per Incoterms 2020)
  - Shipment/Delivery Deadline:
  - Quality & Quantity Standards:
 
  [Actual Situation]
  - Document Discrepancy Table:
    | Item | Contract/L/C Standard | Actual Document Content | Document Name |
    |------|----------------------|------------------------|---------------|
    | ... | ... | ... | ... |
  - Notice Date & Method:
  - Claim Content:
  - Party Arguments: (Seller / Buyer / Bank respectively)
"""

# INPUT_LANG에 따라 P2_SYS의 포맷 블록을 동적으로 교체
P2_FORMAT_LABELS = P2_SYS_EN_LABELS if CONFIG['INPUT_LANG'] == 'en' else P2_SYS_KO_LABELS

In [8]:
P2_SYS = r"""당신은 국제 무역 법률 전문가이자 파인튜닝 데이터 엔지니어입니다.
아래 메타데이터를 기반으로 파인튜닝 샘플의 사실관계(input 필드)만 작성하십시오.

[적용 법령]
{{LAW_LIST}}

[언어 작성 규칙]
{{LANG_CONSTRAINT}}

{{FORMAT_LABELS}}

[공통 작성 규칙]
- "실제 발생 사항" 블록은 포맷 B에만 존재한다. 포맷 A에 절대 포함하지 않는다.
- fallback_level≥2이면: 관련 법령에 완전히 일치하는 조문이 없는 회색지대 상황을 설계하라.
  이는 Prompt 3에서 유추 적용·상위 원칙·실무 관행 추론을 유도하기 위함이다.
- relation_required=true이면: 두 개 이상의 법령이 명시적으로 교차하는 상황을 사실관계에 반영하라.
- verdict가 🟢이면: 모든 조항과 서류가 실제로 적합한 상황을 설계하라. 억지로 위반을 만들지 않는다.
- 수치(금액·부보율·날짜)는 구체적으로 기재하되, 계산 결과를 확정하지 않는다.
- [LAW_LIST]에 없는 법령은 사실관계에 등장시키지 않는다.

[UK 관할 케이스 사실관계 설계 지침 — jurisdiction에 "UK"/"영국" 포함 시 적용]
- 준거법 후보란에 영국법(English law)을 명시하고, 적용 가능한 영국법 시리즈를 괄호로 병기하라.
  예: "준거법 후보: English law (COGSA 1992, SGA 1979 적용 가능)"
- 분쟁해결 조항에 London Court of International Arbitration(LCIA) 또는
  영국 법원(English courts)을 준거법·분쟁해결 조항 후보로 포함시켜라.
- 영국법 시리즈 중 관련 법령을 최소 1개 이상 사실관계에 실질적으로 등장시켜야 한다.
  단, 등장시킬 법령은 반드시 [LAW_LIST]에 포함된 것이어야 한다.
- 한·영 혼합 관할(예: 한국 매도인 ↔ 영국 매수인)의 경우, 두 국가의 강행법규가
  동시에 언급될 수 있는 상황(예: 한국 외국환거래법 + COGSA 1992)을 설계하라.

[출력 형식]
사실관계 텍스트만 출력하십시오. JSON 래핑, 마크다운 코드블록, 설명 문장 금지."""
P2_USR = r"""아래 메타데이터를 기반으로 사실관계(input)를 작성하십시오.

[분량 규칙]
- 계약서 초안 주요 조항: 각 조항을 실제 계약서 수준의 조문 형태로 작성 (조항당 3~5문장)
- 포맷 B 서류 대조표: 최소 6개 항목 이상, 각 서류의 세부 수치(날짜·금액·수량·단위) 모두 명기
- 관계자 항변: 매도인·매수인·은행 각각 2개 이상의 구체적 항변 논거 포함
- 계약 배경 서술: 거래 경위, 협상 경과, 분쟁 촉발 사건을 2~3문단으로 서술

[시나리오 메타데이터]
{{SCENARIO_META}}

위 규칙에 따라 input 텍스트만 출력하십시오."""

P2_SYS = P2_SYS.replace("{{FORMAT_LABELS}}", P2_FORMAT_LABELS)

### 5. [Step 2] 사실관계 생성 실행

In [9]:
async def generate_inputs(target_id=None):
    MAX_RETRIES = 5
    count = 0
    lang_constraint = LANG_CONSTRAINTS.get(CONFIG['INPUT_LANG'], '')
    extra_constraints = CONFIG.get('EXTRA_CONSTRAINTS', '')

    for s_dir in sorted(SOURCE_DIR.iterdir()):
        if not s_dir.is_dir(): continue
        
        # 필터링 로직 추가: target_id가 지정된 경우 폴더명에 해당 ID가 없으면 스킵
        if target_id and target_id not in s_dir.name:
            continue
            
        if (s_dir / "input.txt").exists():
            print(f"⏭️ {s_dir.name} 이미 존재 (스킵)")
            continue
            
        with open(s_dir / "meta.json", "r", encoding="utf-8") as f: meta = json.load(f)
        
        sys = (
            P2_SYS
            .replace("{{LAW_LIST}}", LAW_LIST)
            .replace("{{LANG_CONSTRAINT}}", lang_constraint)
        )
        usr = (
            P2_USR
            .replace("{{SCENARIO_META}}", json.dumps(meta, ensure_ascii=False))
        )
        
        if extra_constraints:
            usr += f"\n\n[추가 제약]\n{extra_constraints}"
            
        for attempt in range(MAX_RETRIES):
            try:
                res = client.models.generate_content(model=MODEL_ID, contents=[sys + "\n" + usr], 
                config = types.GenerateContentConfig(thinking_config=types.ThinkingConfig(thinking_level="high"),
                max_output_tokens=65536
                ))
                text = res.text.strip() if res.text else ""
                if len(text) > 50:
                    with open(s_dir / "input.txt", "w", encoding="utf-8") as f: f.write(text)
                    print(f"✅ {s_dir.name}/input.txt 저장")
                    await asyncio.sleep(1)
                    break
                else:
                    print(f"⚠️ {s_dir.name} 응답 짧음 ({attempt+1}/{MAX_RETRIES})")
                    if attempt < MAX_RETRIES - 1: await asyncio.sleep(2)
            except Exception as e:
                if attempt < MAX_RETRIES - 1:
                    print(f"⚠️ {s_dir.name} 시도 {attempt+1} 실패: {e}. 재시도...")
                    await asyncio.sleep(5)
                else:
                    print(f"❌ {s_dir.name} 최종 실패: {e}")

await generate_inputs(target_id=TARGET_RUN_ID)

✅ b-001_0515_0111/input.txt 저장
✅ b-002_0515_0111/input.txt 저장
✅ b-003_0515_0111/input.txt 저장
✅ b-004_0515_0111/input.txt 저장
⚠️ b-005_0515_0111 시도 1 실패: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}. 재시도...
✅ b-005_0515_0111/input.txt 저장
⚠️ b-006_0515_0111 시도 1 실패: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}. 재시도...
✅ b-006_0515_0111/input.txt 저장
⚠️ b-007_0515_0111 시도 1 실패: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}. 재시도...
⚠️ b-007_0515_0111 시도 2 실패: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing hig

### 6. [Step 3] 법리 판단(Output) 생성 프롬프트 정의

In [ ]:
P3_SYS = r"""당신은 국제 무역 법률 전문가이자 파인튜닝 데이터 엔지니어입니다.
아래 사실관계(input)와 메타데이터를 바탕으로 법리 판단 보고서(output 필드)를 작성하십시오.

[적용 법령]
{{LAW_LIST}}

[언어 작성 규칙]
1. 한국어로 작성한다.
2. 단, 계약서 원문에 다른 언어로 표기된 이름, 주소, 한국법이 아닌 법령 등은 원문과 같이 표기한다.

[핵심 생성 규칙 — 위반 시 샘플 폐기]

규칙 1. 조문 정확성
  - 모든 법령 인용에 Article/조 번호를 명기한다.
  - 조문 번호가 불확실하면 "[조문 확인 필요]" 태그를 붙이되, 추론은 계속 진행한다.
  - "관련 법령에 따르면"으로만 서술하는 것은 금지한다.

규칙 2. 법령 위계 준수 (관할별 1순위 분기 후 공통 순서 적용)

  ※ 관할 판단 우선: jurisdiction 필드의 준거법 또는 분쟁해결 합의를 먼저 확인한다.

  [한국 관할 또는 준거법이 한국법인 경우]
  1순위: 대한민국 강행법규 (대외무역법·외국환거래법·독점규제법·약관규제법 등)
  2순위: 당사자 간 계약 특약 (L/C 조건, 계약서 Annex)
  3순위: 국제조약/협약 (CISG, UCP 600, eUCP 등)
  4순위: 국제 무역 관습 (Incoterms 2020, ISBP 821 등)
  5순위: 임의법규·일반원칙

  [UK 관할 또는 준거법이 영국법(English law)인 경우]
  1순위: 영국 강행법규 (COGSA 1992·SGA 1979·UCTA 1977·Insurance Act 2015 등)
  2순위: 당사자 간 계약 특약 (L/C 조건, 계약서 Annex)
  3순위: 국제조약/협약 (CISG, UCP 600, eUCP 등)
  4순위: 국제 무역 관습 (Incoterms 2020, ISBP 821 등)
  5순위: 임의법규·일반원칙·보통법(Common Law) 원칙

  [한·영 혼합 관할 또는 준거법 합의 불명확 시]
  - 각 쟁점별로 "한국 관할 기준" 또는 "UK 관할 기준" 중 더 유리한 쪽을 명시 후 적용한다.
  - 두 체계의 강행법규가 동시에 적용될 가능성이 있으면 교차 충돌로 처리하고
    관계 유형 태그 [관계 유형: 교차 충돌]을 명시한다.
  - 어느 국가의 강행법규가 1순위인지를 반드시 output에 명시한다.

  충돌이 있으면 우선순위 적용 사실을 output에 반드시 명시한다.

규칙 3. 법령 간 관계 유형 명시 (relation_required=true인 경우 필수)
  - 관계 유형은 조문 적용 섹션의 첫 문장에 온다.
  - 형식: [관계 유형: 적용 제한 / 상위 원칙 포섭 / 교차 충돌 / 요건 보완 / 절차 연동]
  - 단일 법령 케이스도 "단일 법령 적용" 또는 "다른 법령과의 충돌 없음"을 명시한다.
  - 관계 유형 명시 없이 복수 법령을 결합하는 것은 금지한다.

규칙 4. 끈기 있는 폴백 추론 (fallback_level에 따라 적용)
  [1단계] 직접 조문 적용
    서술: "{조문}에 의해 직접 적용된다."
  [2단계] 유사 조문 유추 적용 — [추론 등급: 유사 조문 유추] 필수 명시
    서술: "{조문}은 본 사안을 직접 규율하지 않으나, 동일한 법적 가치인 {가치명}을
          보호하는 조문으로서 유추 적용한다. 유사성 근거: {1문장 이상}"
  [3단계] 상위 법 원칙 적용 — [추론 등급: 상위 원칙 적용] 필수 명시
    서술: "직접 적용 가능한 조문은 확인되지 않으나, {법령명}의 기저 원칙인
          {원칙명}에 근거하여 다음과 같은 리스크가 존재한다고 판단한다."
  [4단계] 실무 관행 기반 — [추론 등급: 실무 관행 기반] 필수 명시, 법적 구속력 없음 고지
    서술: "조문 및 원칙에 의한 확정 판단은 어려우나, {실무 관행/ICC·UNCTAD 가이드라인}에
          따르면 {의견}. [추론 등급: 실무 관행 기반] (법적 구속력 없음)"
  2단계 이상 사용 시 추론 등급 태그 미명시는 금지한다.

규칙 5. 판정 등급 확신도 조정
  - 3·4단계 폴백 추론 사용 시: 동일 사실관계라도 원칙적으로 🟡로 조정한다.
  - 단, 상위 원칙 적용 단계에서도 위반이 명백한 경우 🔴 유지 가능 (근거 명시).
  - 메타데이터의 verdict 값을 최종 판정에 반드시 반영한다.

규칙 6. 생성 금지 패턴
  - 조문 없이 판단 → 금지
  - 복수 법령 결합 시 관계 유형 미명시 → 금지
  - 조문 불일치 상황에서 폴백 없이 "판단 불가" → 금지
  - 수치 계산 결과를 모델이 단독 확정 → 금지 (산식만 제시)
  - 포맷 A에 항변 검토 포함 → 금지
  - 포맷 B에 서류 대조표 없는 불일치 서술 → 금지
  - UK 관할 케이스에서 대한민국 강행법규를 1순위로 적용 → 금지
  - 한국 관할 케이스에서 영국 강행법규를 1순위로 적용 → 금지

[포맷 A output 구조 — format="A"인 경우]
아래 구조를 정확히 따른다. 각 단계는 생략 불가. 해당 없으면 "해당 없음" 명시.

  【계약서 사전 검토 보고서】

  ■ 1단계: 적용 법령 확정
  - 관할 판단: (준거법·분쟁해결 합의 기준으로 한국 관할 / UK 관할 / 혼합 관할 명시)
  - 각 법령의 적용 근거 조문 명시
  - 법령 간 충돌·누락 여부 및 우선순위 적용 사실 명시
  - 복수 법령 결합 시: 관계 유형(규칙 3) 명시

  ■ 2단계: 조항별 리스크 분석

  [조항 A] (조항명 또는 주제)
  - [관계 유형: (해당 시 유형명)] — 단일 법령이면 "단일 법령 적용"
  - 적용 조문: (Article/조 번호 + 조문 요지)
    → 직접 조문 없을 시: [추론 등급: ...] 명시 후 근거 서술
  - 리스크 유형: 무효 가능성 / 집행 불능 / 분쟁 유발 / 불리한 조건
  - 리스크 내용:
  - 수정 권고:
    Before: (현행 조항 요지)
    After:  (권고 조항 요지 + 근거 조문)

  [조항 B] ... (리스크가 있는 조항마다 반복)

  ■ 3단계: 누락 조항 점검
  - (반드시 포함해야 하나 초안에 없는 조항 + 근거 조문)
  - 해당 없으면 "누락 없음"

  ■ 최종 판정
  🔴위험 / 🟡주의 / 🟢적합  (하나만 선택)

  판정 근거 요약:
  - (핵심 리스크 + 판단 결과 bullet 3~5줄)
  - (폴백 추론 사용 시 추론 등급과 확신도 한계 명시)

  【추가 확인 필요 사항】
  - (판단에 영향을 미칠 수 있는 미확인 정보)
  - 해당 없으면 "없음"

[포맷 B output 구조 — format="B"인 경우]
아래 구조를 정확히 따른다. 각 단계는 생략 불가. 해당 없으면 "해당 없음" 명시.

  【무역서류 정합성 검토 보고서】

  ■ 1단계: 적용 법령 확정
  - 관할 판단: (준거법·분쟁해결 합의 기준으로 한국 관할 / UK 관할 / 혼합 관할 명시)
  - 각 법령의 적용 근거 조문 명시
  - 법령 간 충돌 여부 및 우선순위 적용 사실 명시
  - 복수 법령 결합 시: 관계 유형(규칙 3) 명시

  ■ 2단계: 서류별 불일치 특정 및 법적 하자 판단

  [불일치 A] (서류명 — 항목)
  - [관계 유형: (해당 시 유형명)] — 단일 법령이면 "단일 법령 적용"
  - 기준: (계약서·L/C 기재 내용)
  - 실제: (제출 서류 기재 내용)
  - 적용 조문: (Article/조 번호 + 조문 요지)
    → 직접 조문 없을 시: [추론 등급: ...] 명시 후 근거 서술
  - 하자 유형: 본질적 하자 / 비본질적 하자 / 하자 없음
  - 판단 근거: (조문 요건 또는 폴백 추론 근거에 사실관계를 대입한 인과관계 서술)

  [불일치 B] ... (불일치 항목마다 반복)

  ■ 3단계: 관계자 항변 검토
  - 매도인 항변: (항변 내용 → 인용/기각 + 조문 근거)
  - 매수인 항변: (항변 내용 → 인용/기각 + 조문 근거)
  - 은행 항변: (해당 시 / 해당 없으면 생략)

  ■ 4단계: 조치 가능성
  - (행사 가능한 조치 + 조문)
  - 수치가 있는 경우 산식만 제시, 최종값 확정 금지

  ■ 최종 판정
  🔴하자 / 🟡조건부 / 🟢적합  (하나만 선택)

  판정 근거 요약:
  - (핵심 조문 + 판단 결과 bullet 3~5줄)
  - (폴백 추론 사용 시 추론 등급과 확신도 한계 명시)

  【추가 확인 필요 사항】
  - (판단에 영향을 미칠 수 있는 미확인 정보)
  - 해당 없으면 "없음"

[출력 형식]
반드시 아래 형식을 엄수하십시오.
<thought>
위에서 정의한 4단계 폴백 추론(직접->유추->원칙->관행) 과정을 상세히 서술하십시오.
</thought>
<answer>
규정된 보고서 구조(■ 1단계 ~ 【추가 확인 필요 사항】)에 맞춰 최종 보고서를 작성하십시오.
</answer>


[시나리오 메타데이터]
{{SCENARIO_META}}

[분량 기준]
- 각 조항/불일치 항목 분석: 최소 200자 이상
- 수정 권고(Before→After): 조문 원문 인용 + 권고 조항 전문 작성
- 관계자 항변 검토: 각 항변에 대해 인용/기각 근거를 조문 레벨로 상세 서술
- 최종 판정 근거: bullet 5~8줄, 각 bullet은 조문 번호 포함


[사실관계 (input)]
{{INPUT_TEXT}}

위 규칙에 따라 보고서 텍스트만 출력하십시오."""


### 7. [Step 3] 법리 판단 생성 실행

In [11]:
async def generate_outputs(target_id=None):
    MAX_RETRIES = 10
    count = 0
    for s_dir in sorted(SOURCE_DIR.iterdir()):
        if not s_dir.is_dir() or not (s_dir / "input.txt").exists(): continue
        
        # 필터링 로직 추가
        if target_id and target_id not in s_dir.name:
            continue
            
        if (s_dir / "output.txt").exists():
            print(f"⏭️ {s_dir.name} 이미 존재 (스킵)")
            continue

        with open(s_dir / "meta.json", "r", encoding="utf-8") as f: meta = json.load(f)
        with open(s_dir / "input.txt", "r", encoding="utf-8") as f: input_text = f.read()
        
        sys = P3_SYS.replace("{{LAW_LIST}}", LAW_LIST)
        usr = P3_USR.replace("{{SCENARIO_META}}", json.dumps(meta, ensure_ascii=False)).replace("{{INPUT_TEXT}}", input_text)
        
        for attempt in range(MAX_RETRIES):
            try:
                res = client.models.generate_content(model=MODEL_ID, contents=[sys + "\n" + usr], 
                config = types.GenerateContentConfig(thinking_config=types.ThinkingConfig(thinking_level="high"),
                max_output_tokens=65536
                ))
                text = res.text.strip() if res.text else ""
                if len(text) > 100:
                    with open(s_dir / "output.txt", "w", encoding="utf-8") as f: f.write(text)
                    print(f"✅ {s_dir.name}/output.txt 저장")
                    await asyncio.sleep(1)
                    break
                else:
                    print(f"⚠️ {s_dir.name} 응답 짧음 ({attempt+1}/{MAX_RETRIES})")
                    if attempt < MAX_RETRIES - 1: await asyncio.sleep(2)
            except Exception as e:
                if attempt < MAX_RETRIES - 1:
                    print(f"⚠️ {s_dir.name} 시도 {attempt+1} 실패: {e}. 재시도...")
                    await asyncio.sleep(3)
                else:
                    print(f"❌ {s_dir.name} 최종 실패: {e}")

await generate_outputs(target_id=TARGET_RUN_ID)

⚠️ b-001_0515_0111 시도 1 실패: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}. 재시도...
✅ b-001_0515_0111/output.txt 저장
✅ b-002_0515_0111/output.txt 저장
⚠️ b-003_0515_0111 시도 1 실패: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}. 재시도...
✅ b-003_0515_0111/output.txt 저장
⚠️ b-004_0515_0111 시도 1 실패: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}. 재시도...
✅ b-004_0515_0111/output.txt 저장
✅ b-005_0515_0111/output.txt 저장
⚠️ b-006_0515_0111 시도 1 실패: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand

### 8. 최종 데이터 병합

In [17]:
def assemble(target_id=None):
    all_data = []
    collections = {"a": [], "b": []}
    
    # Run ID가 지정된 경우 파일명에 접미사 추가
    suffix = f"_{target_id}" if target_id else ""
    
    for s_dir in sorted(SOURCE_DIR.iterdir()):
        if not s_dir.is_dir(): continue
        
        # 필터링 로직: target_id가 폴더명에 포함되지 않으면 스킵
        if target_id and target_id not in s_dir.name:
            continue
            
        m_p, i_p, o_p = s_dir / "meta.json", s_dir / "input.txt", s_dir / "output.txt"
        # 세 파일이 모두 존재하는 경우만 병합 대상으로 포함
        if not (m_p.exists() and i_p.exists() and o_p.exists()): continue
        
        with open(m_p, "r", encoding="utf-8") as f: meta = json.load(f)
        fmt = meta['format'].lower()
        
        item = {
            "instruction": INSTRUCTION_A if fmt == 'a' else INSTRUCTION_B, 
            "input": open(i_p, encoding='utf-8').read(), 
            "output": open(o_p, encoding='utf-8').read()
        }
        collections[fmt].append(item)
        all_data.append(item)
    
    # 1. 포맷별 개별 저장 (예: trade_law_format_A_0507_1130.jsonl)
    for fmt, data in collections.items():
        if not data: continue
        output_path = SCRIPT_DIR / f"trade_law_format_{fmt.upper()}{suffix}.jsonl"
        with open(output_path, "w", encoding="utf-8") as f:
            for d in data: f.write(json.dumps(d, ensure_ascii=False) + "\n")
        print(f"🚀 {fmt.upper()} 병합 완료: {output_path.name} ({len(data)} samples)")
    
    # 2. 전체 통합 저장 (예: trade_law_finetune_total_0507_1130.jsonl)
    if all_data:
        total_name = f"trade_law_finetune_total{suffix}.jsonl"
        total_path = SCRIPT_DIR / total_name
        with open(total_path, "w", encoding="utf-8") as f:
            for d in all_data: f.write(json.dumps(d, ensure_ascii=False) + "\n")
        print(f"📦 병합 완료: {total_path.name} ({len(all_data)} samples)")
    else:
        print("⚠️ 병합할 데이터가 없습니다. Run ID를 확인해주세요.")

In [18]:
# 실행 시 TARGET_RUN_ID 전달
# 입력 안 할 시 전체 병합
# target_id=TARGET_RUN_ID
assemble(target_id=TARGET_RUN_ID)

🚀 A 병합 완료: trade_law_format_A_0514_2214.jsonl (25 samples)
🚀 B 병합 완료: trade_law_format_B_0514_2214.jsonl (25 samples)
📦 병합 완료: trade_law_finetune_total_0514_2214.jsonl (50 samples)


In [12]:
# 묶고 싶은 target_id 리스트 정의
TARGET_IDS = ["0514_2214", "0514_2339", "0515_0038", "0515_0111"]

def assemble_bundled(target_ids=None):
    all_data = []
    collections = {"a": [], "b": []}
    
    # 결과 파일명에 사용할 접미사 (원하시는 대로 수정 가능합니다)
    suffix = "_16k" 
    
    for s_dir in sorted(SOURCE_DIR.iterdir()):
        if not s_dir.is_dir(): continue
        
        # 필터링 로직: target_ids 리스트 중 하나라도 폴더명에 포함되어 있는지 확인
        if target_ids:
            if not any(tid in s_dir.name for tid in target_ids):
                continue
            
        m_p, i_p, o_p = s_dir / "meta.json", s_dir / "input.txt", s_dir / "output.txt"
        
        # 세 파일이 모두 존재하는 경우만 병합 대상으로 포함
        if not (m_p.exists() and i_p.exists() and o_p.exists()): 
            continue
        
        with open(m_p, "r", encoding="utf-8") as f: 
            meta = json.load(f)
        fmt = meta['format'].lower()
        
        item = {
            "instruction": INSTRUCTION_A if fmt == 'a' else INSTRUCTION_B, 
            "input": open(i_p, encoding='utf-8').read(), 
            "output": open(o_p, encoding='utf-8').read()
        }
        collections[fmt].append(item)
        all_data.append(item)
    
    # 1. 포맷별 개별 저장
    for fmt, data in collections.items():
        if not data: continue
        output_path = SCRIPT_DIR / f"trade_law_format_{fmt.upper()}{suffix}.jsonl"
        with open(output_path, "w", encoding="utf-8") as f:
            for d in data: 
                f.write(json.dumps(d, ensure_ascii=False) + "\n")
        print(f"🚀 {fmt.upper()} 병합 완료: {output_path.name} ({len(data)} samples)")
    
    # 2. 전체 통합 저장
    if all_data:
        total_name = f"trade_law_finetune_total{suffix}.jsonl"
        total_path = SCRIPT_DIR / total_name
        with open(total_path, "w", encoding="utf-8") as f:
            for d in all_data: 
                f.write(json.dumps(d, ensure_ascii=False) + "\n")
        print(f"📦 병합 완료: {total_path.name} ({len(all_data)} samples)")
    else:
        print("⚠️ 병합할 데이터가 없습니다. target_id들을 다시 확인해주세요.")

# 실행
assemble_bundled(target_ids=TARGET_IDS)


🚀 A 병합 완료: trade_law_format_A_16k.jsonl (42 samples)
🚀 B 병합 완료: trade_law_format_B_16k.jsonl (58 samples)
📦 병합 완료: trade_law_finetune_total_16k.jsonl (100 samples)


In [14]:
import json
import re
from pathlib import Path

def inject_thought_tags(input_file, output_file):
    input_path = Path(input_file)
    output_path = Path(output_file)
    
    processed_count = 0
    with open(input_path, "r", encoding="utf-8") as f_in, \
         open(output_path, "w", encoding="utf-8") as f_out:
        
        for line in f_in:
            if not line.strip(): continue
            item = json.loads(line)
            output_text = item.get("output", "")
            
            # 1. <thought> 부분 추출: "■ 최종 판정" 이전까지의 분석 내용을 가져옵니다.
            # (만약 1~3단계만 별도로 요약하고 싶다면 이 로직을 조정할 수 있습니다)
            thought_match = re.search(r"(.*?)■ 최종 판정", output_text, re.S)
            if thought_match:
                thought_content = thought_match.group(1).strip()
            else:
                # 패턴이 맞지 않을 경우 앞부분 50%를 thought로 간주 (폴백)
                thought_content = output_text[:len(output_text)//2].strip()
            
            # 2. <thought> 내용 정제 (참조 파일처럼 [1단계: ...] 형식으로 소폭 변환 가능)
            thought_content = thought_content.replace("■ 1단계:", "[1단계: 직접 조문]")
            thought_content = thought_content.replace("■ 2단계:", "[2단계: 유사 유추]")
            thought_content = thought_content.replace("■ 3단계:", "[3단계: 상위 원칙]")
            
            # 3. 새로운 output 생성
            # <thought>에는 추론 과정을, <answer>에는 전체 보고서를 넣습니다.
            new_output = f"<thought>\n{thought_content}\n</thought>\n<answer>\n{output_text}\n</answer>"
            
            item["output"] = new_output
            f_out.write(json.dumps(item, ensure_ascii=False) + "\n")
            processed_count += 1
            
    print(f"✅ 변환 완료: {output_path.name} ({processed_count} samples)")

# 실행
inject_thought_tags(
    "data/finetune_gen/script/trade_law_finetune_total_16k.jsonl", 
    "data/finetune_gen/script/trade_law_finetune_total_16k_with_thought.jsonl"
)


✅ 변환 완료: trade_law_finetune_total_16k_with_thought.jsonl (100 samples)


### 9. 데이터 통합 및 포맷 변환 (System 필드 추가)
팀원의 데이터와 합치고, 기존 `instruction`을 `system`으로 옮기며 새로운 `instruction`을 부여합니다.

In [13]:
# 새로운 Instruction 정의
system_content = """
당신은 국제 무역 법률 및 관습에 정통한 전문 변호사입니다. 사용자가 제공하는 무역 계약서나 서류를 검토하여 전문 법리 보고서를 작성하십시오.

### 🧠 [분석 가이드라인: <thought> 태그 필수]
모든 답변 전에 반드시 <thought> 태그 내에 다음의 4단계 폴백 추론 과정을 거쳐야 합니다.
1. [1단계: 직접 조문] 해당 상황에 직접 적용되는 Article/조 번호 특정.
2. [2단계: 유사 유추] 직접 조문이 없는 경우 유사 조문 및 법리 유추.
3. [3단계: 상위 원칙] 신의성실, 계약준수, 공서양속 등 법의 일반 원칙 적용.
4. [4단계: 실무 관행] Incoterms, ISBP 등 국제 표준 무역 관행 기반 판단.

[법령 위계 및 관계 설정]
- 위계: [0순위] 대한민국 강행법규 (항시 적용) > [1순위] 영국 강행법규 (준거법이 영국법이거나 영국 관할인 경우에만 적용) > [2순위] 당사자 계약 특약 > [3순위] 국제조약(CISG 등) > [4순위] 무역 관습(Incoterms 등) > [5순위] 일반 원칙.
- 관계 유형 명시: 적용 제한, 교차 충돌, 요건 보완, 절차 연동, 상위 원칙 포섭 중 하나를 선택하여 설명.

### 📝 [보고서 출력 구조: <answer> 태그 필수]

[태스크 A: 계약서 사전 검토] - "아래 계약서 초안의 법적 리스크를 분석하라"는 트리거 시 사용
■ 1단계: 적용 법령 및 관계 확정 (조문 번호 및 관계 유형 필수)
■ 2단계: 조항별 리스크 분석
  - 리스크 유형: 무효 가능성 / 집행 불능 / 분쟁 유발 / 불리한 조건
  - 수정 권고: [Before] 현행 조항 요지 → [After] 권고 조항 + 근거 조문
■ 3단계: 누락 조항 및 특약 점검
■ 최종 판정: 🔴위험(즉시 수정) / 🟡주의(수정 권고) / 🟢적합(현행 유지)
【추가 확인 필요 사항】 - 구체적인 추가 정보 요구 (절대 "없음" 처리 금지)

[태스크 B: 무역서류 정합성 검토] - "아래 무역서류 간 불일치 및 법령 위반 여부를 판단하라"는 트리거 시 사용
■ 1단계: 적용 법령 및 관계 확정 (조문 번호 및 관계 유형 필수)
■ 2단계: 서류별 불일치 특정 및 법적 하자 판단
  - 서류명 / 항목 / 기준값 / 실제값 / 하자 유형(본질적·비본질적·없음)
■ 3단계: 관계자 항변 검토 - 매도인·매수인·은행 각 항변의 인용/기각 및 근거
■ 4단계: 조치 가능성 - 서류 보완(amendment) / 거절(rejection) / 권리 유보
■ 최종 판정: 🔴하자(거절 가능) / 🟡조건부(보완 시 수리) / 🟢적합(하자 없음)
【추가 확인 필요 사항】 - 구체적인 추가 정보 요구 (절대 "없음" 처리 금지)

### ⚠️ [엄격 가드레일]
- 수치 계산: 절대 최종값을 단독 확정하지 말고, 반드시 산식(Formula)을 함께 제시하십시오.
- 다국어 대응: 한국어 또는 영어 질문에 대해 해당 언어의 법리적 뉘앙스를 유지하며 답변하십시오.
- 판단 불가: 1~4단계 폴백 추론을 모두 수행한 후에도 근거가 부족한 경우에만 선언하십시오.
- 답변 내 줄바꿈은 반드시 \\n 으로 처리하십시오.
"""

INSTRUCTION_A = "아래 계약서 초안의 법적 리스크를 분석하라."
INSTRUCTION_B = "아래 무역서류 간 불일치 및 법령 위반 여부를 판단하라."

In [19]:
# [수정 가능] 합칠 파일 이름 및 결과 파일 이름 설정
FILE_NAME_1 = "trade_law_finetune_total_16k_with_thought.jsonl"
FILE_NAME_2 = "Legal_Agent_Gemma_Finetune_gemma4_e2b_진짜최종.jsonl"
FINAL_OUTPUT_NAME = "Gemma_Finetune_gemma4_e2b_ultimate_final.jsonl"

# [수정 가능] 새롭게 부여할 인스트럭션 내용
INSTR_A = "아래 계약서 초안의 법적 리스크를 분석하라."
INSTR_B = "아래 무역서류 간 불일치 및 법령 위반 여부를 판단하라."

def merge_and_transform():
    input_paths = [SCRIPT_DIR / FILE_NAME_1, SCRIPT_DIR / FILE_NAME_2]
    output_path = SCRIPT_DIR / FINAL_OUTPUT_NAME
    
    processed_data = []
    
    for p in input_paths:
        if not p.exists():
            print(f"⚠️ 파일 없음: {p.name} (스킵)")
            continue
            
        with open(p, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip(): continue
                item = json.loads(line)
                
                # 1. 이미 'text' 필드가 있는 경우 (이미 완성된 포맷)
                if "text" in item:
                    processed_data.append(item)
                    continue
                
                # 2. 'text' 필드가 없는 경우 (변환이 필요한 포맷)
                old_instr = item.get("instruction", "")
                input_text = item.get("input") or item.get("instruction") or ""
                output_text = item.get("output", "")
                
                if "계약서 초안" in old_instr or "계약서" in str(input_text):
                    new_instr = INSTR_A
                else:
                    new_instr = INSTR_B

                user_content = "[지시사항]\n" + new_instr + "\n\n[입력 데이터]\n" + str(input_text)
                
                # 새로운 ChatML 포맷으로 결합
                text_content = (
                    "<|im_start|>system\n" + system_content + "<|im_end|>\n" +
                    "<|im_start|>user\n" + user_content + "<|im_end|>\n" +
                    "<|im_start|>assistant\n" + str(output_text) + "<|im_end|>"
                )
                
                processed_data.append({"text": text_content})
    
    if processed_data:
        with open(output_path, "w", encoding="utf-8") as f:
            for d in processed_data: 
                f.write(json.dumps(d, ensure_ascii=False) + "\n")
        print(f"✅ 최종 병합 및 변환 완료: {output_path.name} ({len(processed_data)} samples)")
    else:
        print("⚠️ 처리할 데이터가 없습니다.")

merge_and_transform()


✅ 최종 병합 및 변환 완료: Gemma_Finetune_gemma4_e2b_ultimate_final.jsonl (865 samples)
